# EUR/USD M15 — Prediksi Next-Bar Return (Versi Diperbaiki)

Notebook ini adalah revisi dari `ML_EURUSD_MLR_Prediction.ipynb`. Perubahan utama:

| Masalah di versi lama | Perbaikan di notebook ini |
|---|---|
| Target = level harga (non-stasioner, R² 0.9998 menyesatkan) | Target = **return bar berikutnya (pips)** + arah naik/turun |
| GD belum konvergen (OHLC kolinear), MLR kalah dari naive | Diagnosa + pembanding closed-form (`lstsq`) |
| Hanya 4 fitur OHLC | ~35 fitur: momentum, candle, volatilitas, mean-reversion, volume, spread, sesi |
| Satu split 70/30 | **Walk-forward** 5 fold (expanding window) |
| Hanya RMSE/MAE/MAPE/R² level | RMSE pips, R² OOS vs naive, **directional accuracy**, uji Diebold-Mariano & binomial |
| Tidak ada backtest | **Backtest dengan spread** + sweep threshold keyakinan |
| `shift(-1)` melewati gap weekend | Baris dengan gap waktu ≠ 15 menit dibuang dari target |
| Scaler disimpan dari seluruh data (tidak cocok dengan model) | `Pipeline` (scaler + model) disimpan bersama daftar fitur |

**Struktur folder:** jalankan dari folder `notebooks/`; data dibaca dari `../data/raw/` dan model disimpan ke `../models/` (lihat README). Di Colab, cukup upload `EURUSD_M15.csv` ke direktori kerja.

## 1. Import & Konfigurasi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib
from scipy import stats

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

import os
CSV_PATH  = next((p for p in ['../data/raw/EURUSD_M15.csv', 'data/raw/EURUSD_M15.csv', 'EURUSD_M15.csv'] if os.path.exists(p)), 'EURUSD_M15.csv')
MODEL_DIR = '../models' if os.path.isdir('../models') else '.'      # folder output model
PIP      = 1e-4          # 1 pip EUR/USD
SEED     = 42
np.random.seed(SEED)

## 2. Load Data

File CSV punya header 6 nama kolom tetapi baris datanya 7 field (Time, Open, High, Low, Close, Volume, Spread),
sehingga pandas otomatis menjadikan kolom pertama (waktu) sebagai index. Kita rename kolomnya secara eksplisit.

In [ ]:
raw = pd.read_csv(CSV_PATH, sep='\t')
if raw.shape[1] == 6:                      # waktu menjadi index otomatis
    raw.columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'Spread']
    raw.index = pd.to_datetime(raw.index)
else:                                      # header lengkap
    raw['Time'] = pd.to_datetime(raw['Time'])
    raw = raw.set_index('Time')[['Open', 'High', 'Low', 'Close', 'Volume', 'Spread']]
raw.index.name = 'Datetime'
raw = raw.sort_index()
raw = raw[~raw.index.duplicated()]

df = raw.copy()
print('Shape   :', df.shape)
print('Rentang :', df.index[0], '->', df.index[-1])
print('Missing :', int(df.isnull().sum().sum()))
df.head()

## 3. Data Quality & EDA yang Relevan

Pada deret harga, histogram level dan korelasi OHLC tidak informatif (harga non-stasioner, OHLC pasti berkorelasi ≈ 1).
Yang perlu dilihat: gap waktu, distribusi **return**, autokorelasi return, stasioneritas, dan pola volatilitas per jam.

In [ ]:
dt = df.index.to_series().diff()
n_gap = (dt > pd.Timedelta('15min')).sum()
print(f'Gap > 15 menit : {n_gap}  (weekend/libur; `shift(-1)` di sini BUKAN 15 menit ke depan)')
print(f'Gap terbesar   : {dt.max()}')

ret = df['Close'].diff() / PIP
ret[dt != pd.Timedelta('15min')] = np.nan          # return lintas gap tidak dipakai

print(f'\nRata-rata |return| : {ret.abs().mean():.2f} pips')
print(f'Rata-rata range    : {((df.High-df.Low)/PIP).mean():.2f} pips')
print(f'Rata-rata spread   : {(df.Spread/10).mean():.2f} pips  (asumsi 1 point = 0.1 pip, broker 5-digit)')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# (a) distribusi return vs normal
r = ret.dropna()
axes[0,0].hist(r.clip(-30, 30), bins=120, density=True, alpha=0.7)
xs = np.linspace(-30, 30, 400)
axes[0,0].plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), 'r--', label='Normal')
axes[0,0].set_title(f'Distribusi return M15 (pips) — kurtosis={stats.kurtosis(r):.1f} (fat tail)')
axes[0,0].legend()

# (b) ACF return
acf = [r.autocorr(k) for k in range(1, 31)]
axes[0,1].bar(range(1, 31), acf)
band = 1.96 / np.sqrt(len(r))
axes[0,1].axhline(band, color='r', ls='--'); axes[0,1].axhline(-band, color='r', ls='--')
axes[0,1].set_title('ACF return (lag 1–30): hampir tidak ada struktur linear')

# (c) ACF |return| (volatility clustering)
acf_abs = [r.abs().autocorr(k) for k in range(1, 31)]
axes[1,0].bar(range(1, 31), acf_abs, color='darkorange')
axes[1,0].set_title('ACF |return|: volatility clustering (ini yang bisa diprediksi)')

# (d) volatilitas per jam
by_hour = r.abs().groupby(r.index.hour).mean()
axes[1,1].bar(by_hour.index, by_hour.values, color='seagreen')
axes[1,1].set_title('Rata-rata |return| per jam (waktu data)'); axes[1,1].set_xlabel('Jam')
plt.tight_layout(); plt.show()

print('Autokorelasi Close lag-1       :', round(df.Close.autocorr(1), 5), '(non-stasioner)')
print('Autokorelasi return lag-1      :', round(r.autocorr(1), 5))
try:
    from statsmodels.tsa.stattools import adfuller
    print('ADF p-value Close  :', round(adfuller(df.Close.values, maxlag=10)[1], 4))
    print('ADF p-value Return :', round(adfuller(r.values, maxlag=10)[1], 4))
except ImportError:
    print('statsmodels tidak terpasang — lewati ADF')

## 4. Diagnosa Notebook Lama: Kenapa MLR Kalah dari Naive?

Kita replikasi setup lama (fitur OHLC level, target Close t+1, min-max, gradient descent) lalu bandingkan dengan solusi **closed-form** OLS.

In [ ]:
class MultipleLinearRegression:
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.lr, self.n_iter = learning_rate, n_iterations
    def fit(self, X, y):
        n, k = X.shape
        self.w, self.b, self.loss = np.zeros(k), 0.0, []
        for _ in range(self.n_iter):
            err = X @ self.w + self.b - y
            self.loss.append(np.mean(err**2))
            self.w -= self.lr * (2/n) * (X.T @ err)
            self.b -= self.lr * (2/n) * err.sum()
        return self
    def predict(self, X): return X @ self.w + self.b

lv = df[['Open','High','Low','Close']].copy()
lv['y'] = lv['Close'].shift(-1)
lv = lv.dropna()
cut = int(len(lv) * 0.7)
tr, te = lv.iloc[:cut], lv.iloc[cut:]
fc = ['Open','High','Low','Close']

mn, mx = tr[fc].min(), tr[fc].max()
ymn, ymx = tr['y'].min(), tr['y'].max()
Xtr = ((tr[fc]-mn)/(mx-mn)).values;  ytr = ((tr['y']-ymn)/(ymx-ymn)).values
Xte = ((te[fc]-mn)/(mx-mn)).values

gd = MultipleLinearRegression(0.1, 1000).fit(Xtr, ytr)
p_gd = gd.predict(Xte) * (ymx-ymn) + ymn

ols = LinearRegression().fit(tr[fc], tr['y'])
p_ols = ols.predict(te[fc])

rmse = lambda a, b: np.sqrt(np.mean((np.asarray(a)-np.asarray(b))**2))
print(f"RMSE Naive (Close t)    : {rmse(te['y'], te['Close'])/PIP:.4f} pips")
print(f"RMSE MLR gradient desc. : {rmse(te['y'], p_gd)/PIP:.4f} pips   <- bobot GD (scaled): {gd.w.round(3)}")
print(f"RMSE OLS closed-form    : {rmse(te['y'], p_ols)/PIP:.4f} pips   <- koef (Open,High,Low,Close): {ols.coef_.round(3)}")
print(f"\nCondition number X'X (scaled): {np.linalg.cond(Xtr.T @ Xtr):,.0f}  -> multikolinearitas ekstrem, GD sangat lambat konvergen")

**Kesimpulan diagnosa:** OLS yang dioptimasi dengan benar menghasilkan koefisien Close ≈ 0.93 (jumlah koefisien ≈ 1) dan RMSE **sama** dengan naive.
Artinya model level "terlihat" kalah bukan karena data, tapi karena GD tertahan di lembah sempit akibat OHLC yang hampir identik.
Dan kalaupun benar, model level paling banter *menyamai* naive — tidak ada informasi baru yang ditambahkan.

## 5. Feature Engineering (Target = Return Bar Berikutnya)

Semua fitur hanya memakai informasi sampai bar `t` (tidak ada look-ahead). Sesi diasumsikan dari jam pada data
(weekend open Minggu ±21:00 mengindikasikan waktu UTC); sesuaikan batas jam jika waktu server broker kamu berbeda.

In [ ]:
def build_features(d):
    """d: DataFrame OHLCV+Spread ber-index waktu. Return: DataFrame fitur (hanya info <= t)."""
    c = d['Close']
    f = pd.DataFrame(index=d.index)
    ret1 = c.diff() / PIP

    # 1) Momentum / return multi-horizon (15m, 30m, 45m, 1h15, 2h30, 5h, 12h, 24h)
    for k in (1, 2, 3, 5, 10, 20, 48, 96):
        f[f'ret{k}'] = c.diff(k) / PIP

    # 2) Struktur candle
    rng = (d.High - d.Low)
    f['body']       = (c - d.Open) / PIP
    f['range']      = rng / PIP
    f['upper_wick'] = (d.High - d[['Open', 'Close']].max(axis=1)) / PIP
    f['lower_wick'] = (d[['Open', 'Close']].min(axis=1) - d.Low) / PIP
    f['close_pos']  = ((c - d.Low) / rng.replace(0, np.nan)).fillna(0.5)

    # 3) Volatilitas
    f['vol20']     = ret1.rolling(20).std()
    f['vol96']     = ret1.rolling(96).std()
    f['vol_ratio'] = f['vol20'] / f['vol96']
    tr_ = pd.concat([rng, (d.High - c.shift()).abs(), (d.Low - c.shift()).abs()], axis=1).max(axis=1) / PIP
    f['atr14']     = tr_.rolling(14).mean()

    # 4) Mean-reversion / tren (dinormalisasi volatilitas)
    f['dev_sma20_z'] = (c - c.rolling(20).mean())  / PIP / f['vol96']
    f['dev_sma96_z'] = (c - c.rolling(96).mean())  / PIP / f['vol96']
    f['sma20_slope'] = (c.rolling(20).mean() - c.rolling(20).mean().shift(5)) / PIP
    delta = c.diff()
    up = delta.clip(lower=0).rolling(14).mean(); dn = (-delta.clip(upper=0)).rolling(14).mean()
    f['rsi14'] = 100 - 100 / (1 + up / dn.replace(0, np.nan))

    # 5) Volume & spread (likuiditas)
    f['logvol']     = np.log1p(d['Volume'])
    f['volume_z96'] = (d['Volume'] - d['Volume'].rolling(96).mean()) / d['Volume'].rolling(96).std()
    f['spread_pips'] = d['Spread'] / 10.0
    f['spread_z96']  = (d['Spread'] - d['Spread'].rolling(96).mean()) / d['Spread'].rolling(96).std()

    # 6) Waktu & sesi
    h = d.index.hour + d.index.minute / 60
    f['hour_sin'] = np.sin(2*np.pi*h/24); f['hour_cos'] = np.cos(2*np.pi*h/24)
    f['dow']      = d.index.dayofweek
    f['sess_asia']    = (d.index.hour < 7).astype(int)
    f['sess_london']  = ((d.index.hour >= 7)  & (d.index.hour < 16)).astype(int)
    f['sess_ny']      = ((d.index.hour >= 13) & (d.index.hour < 22)).astype(int)
    f['sess_overlap'] = ((d.index.hour >= 13) & (d.index.hour < 16)).astype(int)

    return f.replace([np.inf, -np.inf], np.nan)

F = build_features(df)

# Target: return bar berikutnya (pips); hanya valid jika bar berikutnya tepat 15 menit setelahnya
next_ok = (df.index.to_series().shift(-1) - df.index.to_series()) == pd.Timedelta('15min')
y = ((df['Close'].shift(-1) - df['Close']) / PIP).where(next_ok)

data = F.assign(y=y, spread_bt=df['Spread'] / 10.0).dropna()
FEATS = list(F.columns)
print(f'Jumlah fitur : {len(FEATS)}')
print(f'Sampel valid : {len(data):,}  (dari {len(df):,})')
print(f'Target y     : mean={data.y.mean():+.4f} pips, std={data.y.std():.3f} pips, P(y>0)={np.mean(data.y>0):.3f}')

In [ ]:
# Korelasi tiap fitur dengan target (sinyal sangat kecil = wajar di pasar efisien)
corr = data[FEATS].corrwith(data['y']).sort_values(key=np.abs, ascending=False)
plt.figure(figsize=(8, 7))
corr.head(20)[::-1].plot.barh(color='steelblue')
plt.title('20 fitur dengan |korelasi| tertinggi terhadap return t+1')
plt.axvline(0, color='k', lw=0.8); plt.tight_layout(); plt.show()
print(corr.head(8).round(4))

## 6. Walk-Forward Validation

Expanding window: train mulai dari 40% data awal, lalu 5 fold test berurutan (masing-masing ±12% data) tanpa overlap.
Setiap fold melatih ulang model hanya dengan data sebelum fold test tersebut.

Model yang dibandingkan: **Naive (prediksi 0)**, **rata-rata train**, **AR(1)** (hanya `ret1`), **Ridge**, **HistGradientBoosting**
(regresor), dan **HistGB Classifier** (arah naik/turun).

In [ ]:
N_FOLDS, START = 5, 0.40

REG_SPECS = {
    'AR(1) ret1':       (['ret1'], lambda: LinearRegression()),
    'Ridge (semua)':    (FEATS,    lambda: make_pipeline(StandardScaler(), Ridge(alpha=100.0))),
    'HistGB (semua)':   (FEATS,    lambda: HistGradientBoostingRegressor(
                                        max_depth=3, learning_rate=0.05, max_iter=200,
                                        min_samples_leaf=300, l2_regularization=5.0, random_state=SEED)),
}
CLF_NAME = 'HistGB-Clf (arah)'

def walk_forward(data):
    n = len(data); step = int(n * (1 - START) / N_FOLDS)
    parts = []
    for k in range(N_FOLDS):
        tr_end = int(n * START) + k * step
        te_end = n if k == N_FOLDS - 1 else tr_end + step
        trn, tst = data.iloc[:tr_end], data.iloc[tr_end:te_end]
        out = pd.DataFrame({'y': tst['y'], 'fold': k + 1, 'spread_bt': tst['spread_bt']}, index=tst.index)
        out['Naive (0)']   = 0.0
        out['Mean train']  = trn['y'].mean()
        for name, (cols, mk) in REG_SPECS.items():
            m = mk().fit(trn[cols], trn['y'])
            out[name] = m.predict(tst[cols])
        # Klasifikasi arah (abaikan bar flat saat training)
        nz = trn['y'] != 0
        clf = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                             min_samples_leaf=300, l2_regularization=5.0, random_state=SEED)
        clf.fit(trn.loc[nz, FEATS], (trn.loc[nz, 'y'] > 0).astype(int))
        out[CLF_NAME] = clf.predict_proba(tst[FEATS])[:, 1] - 0.5      # >0 = prediksi naik
        parts.append(out)
        print(f'Fold {k+1}: train={len(trn):>6,} ({trn.index[0].date()}→{trn.index[-1].date()})  '
              f'test={len(tst):>6,} ({tst.index[0].date()}→{tst.index[-1].date()})')
    return pd.concat(parts)

oof = walk_forward(data)

## 7. Evaluasi

- **RMSE (pips)** dan **R² OOS** relatif terhadap naive: `1 − SSE_model / SSE_naive` (positif = lebih baik dari naive).
- **Directional accuracy** pada bar yang tidak flat, beserta p-value uji binomial (H0: 50%).
- **Diebold-Mariano** (loss kuadrat) vs naive: `t > 0` = model lebih baik; p-value dua sisi.

In [ ]:
def summarize(oof):
    y = oof['y'].values; n = len(y)
    sse0 = np.sum(y**2)
    rows = []
    for col in [c for c in oof.columns if c not in ('y', 'fold', 'spread_bt')]:
        p = oof[col].values
        is_clf = (col == CLF_NAME)
        row = {'Model': col}
        if not is_clf:
            e2 = (y - p)**2
            row['RMSE (pips)'] = np.sqrt(e2.mean())
            row['R2 OOS vs naive'] = 1 - e2.sum() / sse0
            if col != 'Naive (0)':
                d = y**2 - e2
                t = d.mean() / (d.std(ddof=1) / np.sqrt(n))
                row['DM t'] = t; row['DM p'] = 2 * (1 - stats.norm.cdf(abs(t)))
        nz = (y != 0) & (np.sign(p) != 0)
        if nz.sum() > 0 and col != 'Naive (0)':
            hits = int((np.sign(p[nz]) == np.sign(y[nz])).sum())
            row['Dir acc'] = hits / nz.sum()
            row['Binom p'] = stats.binomtest(hits, int(nz.sum()), 0.5).pvalue
        rows.append(row)
    return pd.DataFrame(rows).set_index('Model')

res = summarize(oof)
pd.options.display.float_format = '{:.5f}'.format
res

In [ ]:
# Stabilitas directional accuracy per fold
def dir_acc(g, col):
    nz = (g['y'] != 0) & (np.sign(g[col]) != 0)
    return (np.sign(g.loc[nz, col]) == np.sign(g.loc[nz, 'y'])).mean()

cols_dir = ['AR(1) ret1', 'Ridge (semua)', 'HistGB (semua)', CLF_NAME]
fold_tbl = pd.DataFrame({c: oof.groupby('fold').apply(lambda g: dir_acc(g, c)) for c in cols_dir})
fold_tbl.loc['mean'] = fold_tbl.mean(); fold_tbl.loc['std'] = fold_tbl.iloc[:-1].std()
display(fold_tbl.round(4))

ax = fold_tbl.iloc[:-2].plot(marker='o', figsize=(9, 4))
ax.axhline(0.5, color='k', ls='--', lw=1); ax.set_ylabel('Directional accuracy')
ax.set_title('Directional accuracy per fold walk-forward (garis = 50%)'); plt.show()

## 8. Backtest dengan Biaya (Spread)

Asumsi: entry di Close bar `t` sesuai tanda prediksi, exit di Close bar `t+1`. Biaya = **setengah spread per unit perubahan posisi**
(sehingga satu round-trip = 1 spread penuh; flip arah = 1 spread) + slippage opsional. Spread diambil dari kolom `Spread` pada bar saat transaksi.

Agar tidak hanya trading tiap bar, kita sweep **filter keyakinan**: hanya trade jika |prediksi| berada di atas persentil `q` *dalam fold yang sama*
(tidak ada look-ahead antar fold).

In [ ]:
SLIPPAGE = 0.0   # pips per unit perubahan posisi (tambahkan mis. 0.1–0.3 untuk uji sensitivitas)

def backtest(oof, col, q=0.0, slippage=SLIPPAGE):
    pred = oof[col]
    rank = pred.abs().groupby(oof['fold']).rank(pct=True)
    pos  = np.where(rank.values > q, np.sign(pred.values), 0.0)
    prev = np.r_[0.0, pos[:-1]]
    gross = pos * oof['y'].values
    cost  = np.abs(pos - prev) * (oof['spread_bt'].values / 2 + slippage)
    net   = gross - cost
    in_mkt = pos != 0
    entries = int(((pos != prev) & (pos != 0)).sum())
    return {
        'q': q, 'bars in mkt': int(in_mkt.sum()), 'entries': entries,
        'hit rate': float((gross[in_mkt] > 0).sum() / max(1, (gross[in_mkt] != 0).sum())),
        'gross pips/bar': gross.sum() / max(1, in_mkt.sum()),
        'net pips/bar':   net.sum()   / max(1, in_mkt.sum()),     # biaya exit ikut dihitung
        'net pips/entry': net.sum()   / max(1, entries),
        'total net pips': net.sum(),
    }, pd.Series(net.cumsum(), index=oof.index)

models_bt = ['AR(1) ret1', 'Ridge (semua)', 'HistGB (semua)', CLF_NAME]
rows = []
for m in models_bt:
    for q in (0.0, 0.5, 0.8, 0.9, 0.95, 0.99):
        r_, _ = backtest(oof, m, q); r_['Model'] = m; rows.append(r_)
bt = pd.DataFrame(rows).set_index(['Model', 'q'])
display(bt.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for m in models_bt:
    _, eq = backtest(oof, m, q=0.0);  axes[0].plot(eq.index, eq.values, label=m, lw=0.9)
    _, eq = backtest(oof, m, q=0.90); axes[1].plot(eq.index, eq.values, label=m, lw=0.9)
for ax, t in zip(axes, ['Trade setiap bar (q=0)', 'Hanya 10% sinyal paling yakin (q=0.90)']):
    ax.axhline(0, color='k', lw=0.8); ax.set_title(f'Equity bersih (pips) — {t}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Audit: dari mana "akurasi tinggi" pada sinyal paling yakin?

Classifier terlihat sangat akurat di persentil teratas. Cek dulu **kapan** sinyal itu muncul dan berapa spread-nya sebelum percaya.

In [ ]:
rank = oof[CLF_NAME].abs().groupby(oof['fold']).rank(pct=True)
rows = []
for lo, hi in [(0, .5), (.5, .9), (.9, .99), (.99, 1.0)]:
    s_ = oof[(rank > lo) & (rank <= hi)]; nz = s_['y'] != 0
    rows.append({'bucket keyakinan': f'{lo:.0%}-{hi:.0%}', 'n': len(s_),
                 'dir acc': (np.sign(s_.loc[nz, CLF_NAME]) == np.sign(s_.loc[nz, 'y'])).mean(),
                 'rata2 spread (pips)': s_['spread_bt'].mean(),
                 '% di jam 21-23': np.mean(s_.index.hour >= 21)})
display(pd.DataFrame(rows).set_index('bucket keyakinan').round(4))

ok_ = oof['spread_bt'] <= 1.0
nz = ok_ & (oof['y'] != 0)
print(f"Dir acc classifier hanya pada bar dengan spread <= 1 pip: "
      f"{(np.sign(oof.loc[nz, CLF_NAME]) == np.sign(oof.loc[nz, 'y'])).mean():.4f}  "
      f"(membuang {1 - ok_.mean():.1%} bar)")

## 9. Feature Importance (Permutation, pada Fold Terakhir)

Permutation importance diukur dari kenaikan RMSE saat sebuah fitur diacak — hanya informatif jika modelnya memang punya sinyal;
di sini dipakai untuk melihat fitur mana yang paling dipakai model.

In [ ]:
from sklearn.inspection import permutation_importance
n = len(data); step = int(n * (1 - START) / N_FOLDS)
tr_end = int(n * START) + (N_FOLDS - 1) * step
trn, tst = data.iloc[:tr_end], data.iloc[tr_end:]
m = REG_SPECS['HistGB (semua)'][1]().fit(trn[FEATS], trn['y'])
pi = permutation_importance(m, tst[FEATS], tst['y'], n_repeats=3, random_state=SEED,
                            scoring='neg_root_mean_squared_error')
imp = pd.Series(pi.importances_mean, index=FEATS).sort_values()
plt.figure(figsize=(8, 8)); imp.tail(15).plot.barh(color='teal')
plt.title('Permutation importance (kenaikan RMSE, pips) — HistGB fold terakhir')
plt.tight_layout(); plt.show()

## 10. Simpan Model (Pipeline + Daftar Fitur) & Prediksi Baru

Berbeda dengan versi lama, scaler dan model disimpan bersatu dalam satu `Pipeline` supaya tidak bisa tidak cocok.
Model final dilatih ulang dengan seluruh data.

In [ ]:
final_cols, final_mk = REG_SPECS['Ridge (semua)']
final_model = final_mk().fit(data[final_cols], data['y'])

joblib.dump({'model': final_model, 'features': final_cols, 'target': 'next-bar return (pips)',
             'pip': PIP, 'trained_until': str(data.index[-1])}, f'{MODEL_DIR}/eurusd_m15_ridge.joblib')

def predict_next_return(recent_df, bundle_path=f'{MODEL_DIR}/eurusd_m15_ridge.joblib'):
    """recent_df: minimal ~120 bar M15 terakhir (kolom Open, High, Low, Close, Volume, Spread; index waktu).
       Return: prediksi return bar berikutnya dalam pips (ekspektasi: nilainya sangat kecil)."""
    b = joblib.load(bundle_path)
    feats = build_features(recent_df).iloc[[-1]][b['features']]
    if feats.isnull().any(axis=None):
        raise ValueError('Data terlalu pendek / ada NaN pada fitur — berikan ≥ 120 bar.')
    return float(b['model'].predict(feats)[0])

print('Contoh prediksi (pips) dari 200 bar terakhir:', round(predict_next_return(df.tail(200)), 4))

### Ekspor parameter untuk web (`models/ridge_params.json`)

Browser tidak bisa membaca `.joblib`, jadi koefisien Ridge, statistik scaler, serta tabel keyakinan (dari prediksi out-of-sample walk-forward)
diekspor ke JSON. Setelah file ini terbentuk, jalankan `python src/update_web.py` untuk menyuntikkannya ke `web/index.html`.

In [ ]:
import json
sc, rg = final_model.steps[0][1], final_model.steps[1][1]
p_oos, y_oos = oof['Ridge (semua)'], oof['y']

q  = [round(float(np.quantile(p_oos.abs(), k / 100)), 5) for k in range(101)]   # kuantil |prediksi| OOS
rk = p_oos.abs().rank(pct=True)
acc = []                                                                         # akurasi arah per desil keyakinan
for d in range(10):
    m = (rk > d / 10) & (rk <= (d + 1) / 10) & (y_oos != 0)
    acc.append(round(float((np.sign(p_oos[m]) == np.sign(y_oos[m])).mean()), 4))

params = {'features': final_cols, 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
          'coef': rg.coef_.tolist(), 'b': float(rg.intercept_), 'q': q, 'acc': acc}
with open(f'{MODEL_DIR}/ridge_params.json', 'w') as fh:
    json.dump(params, fh, separators=(',', ':'))
print('Tersimpan:', f'{MODEL_DIR}/ridge_params.json', '| fitur:', len(final_cols), '| acc desil:', acc)

## Kesimpulan (hasil run di data ini)

**Akurasi titik (RMSE) — tidak ada model yang mengalahkan naive.** Pada 5 fold walk-forward (±59.8 ribu bar OOS), RMSE naive = 5.1303 pips, sedangkan semua model berada di 5.1299–5.1317 pips. R² OOS terhadap naive berkisar −0.0005 s/d +0.0002 dan uji Diebold-Mariano tidak signifikan (p = 0.37–0.94). Menambah fitur dari 4 menjadi 32 tidak mengubah ini.

**Arah — ada sinyal sangat tipis, tapi konsisten.** Directional accuracy 50.6–50.8% (AR(1), Ridge, HistGB regresor) dan **51.5%** untuk HistGB Classifier, dengan hasil 51.3–51.95% di kelima fold. Dengan ±60 ribu sampel, selisih 1.5 poin dari 50% itu signifikan secara statistik (binomial p < 0.001), tapi sangat kecil secara ekonomi.

**Backtest — tidak ada konfigurasi yang untung setelah spread.** Dari 24 kombinasi model × filter keyakinan, semuanya menghasilkan total net pips negatif (yang paling mendekati nol: HistGB regresor q=0.99, −9 pips dari 486 entry). Rata-rata keuntungan kotor per bar (±0.05–0.5 pips) lebih kecil dari biaya transaksi (setengah spread ≈ 0.24 pips per perubahan posisi, belum termasuk slippage).

**Hati-hati dengan "akurasi 71%".** Pada 1% sinyal paling yakin, classifier benar 71.6%, tetapi 94% sinyal itu jatuh di jam 21–23 (rollover/likuiditas tipis) dengan spread rata-rata 2.77 pips (vs 0.43 pips rata-rata keseluruhan). Itu pola *bid-ask bounce* di bar ber-spread lebar, bukan edge yang bisa dieksekusi — net-nya −0.89 pips/bar. Jika bar dengan spread > 1 pip dibuang, akurasi classifier turun ke 51.1%.

**Yang bisa diklaim dengan jujur:** (1) model level OHLC versi lama kalah dari naive karena GD tidak konvergen, bukan karena data; (2) dengan target return dan evaluasi walk-forward yang benar, fitur teknikal M15 hanya memberi edge arah ±1–1.5 poin persentase; (3) edge itu habis oleh spread. Angka bisa sedikit berbeda di Colab (versi library), tetapi kesimpulan kualitatifnya seharusnya sama.


## 11. Ide Pengembangan Berikutnya

- **Multi-timeframe** yang benar: resample ke H1/H4/D1 dengan `closed='left', label='left'` lalu `shift(1)` sebelum di-merge ke M15 (hindari look-ahead).
- **Event kalender ekonomi** (NFP, CPI, FOMC, ECB) sebagai fitur "menit ke event" dan flag jam rilis — volatilitas berubah drastis di sekitar event.
- **Data eksternal:** DXY, selisih yield US–DE 2Y, VIX, harga emas/minyak.
- **Prediksi volatilitas** (GARCH/HAR-RV/GBM pada |return| atau range) — target ini jauh lebih bisa diprediksi dari arah (lihat ACF |return|).
- **Label triple-barrier / meta-labeling** (Lopez de Prado) dan **purged k-fold** untuk evaluasi yang lebih ketat.
- **Model sekuensial** (GRU/LSTM, TCN) — hanya setelah baseline tabular di atas terbukti punya edge.